In [1]:
# 06a-1. 기본 설정

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


PROJECT_ROOT = Path(
    r"C:\code\portfolio_optimization"
)


SUPERVISED_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "common"
    / "supervised_dataset.parquet"
)


print(
    "dataset exists:",
    SUPERVISED_DATASET_PATH.exists()
)

dataset exists: True


In [2]:
# 06a-2. supervised dataset 불러오기

supervised_dataset = (
    pq.read_table(
        SUPERVISED_DATASET_PATH
    )
    .to_pandas()
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "shape:",
    supervised_dataset.shape
)

print(
    "signals:",
    supervised_dataset[
        "signal_date"
    ].nunique()
)

print(
    "start:",
    supervised_dataset[
        "signal_date"
    ].min()
)

print(
    "end:",
    supervised_dataset[
        "signal_date"
    ].max()
)

shape: (21779, 39)
signals: 436
start: 2018-05-04 00:00:00
end: 2026-09-04 00:00:00


In [3]:
# 06a-3. model feature 설정

ASSET_FEATURES = [
    "return_1d",
    "return_5d",
    "momentum_20d",
    "momentum_60d",
    "volatility_20d",
    "drawdown_20d",
    "trading_value_ma20",
    "trading_value_ratio_20d",
    "log_market_cap"
]

MARKET_FEATURES = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_volatility_20d",
    "market_drawdown",
    "volume_change_1d",
    "trading_value_change_1d",
    "market_trading_value_ratio_20d"
]

MACRO_FEATURES = [
    "base_rate",
    "usdkrw",
    "bond3y",
    "usdkrw_return_1d",
    "usdkrw_return_5d",
    "usdkrw_return_20d",
    "bond3y_change_1d",
    "bond3y_change_5d",
    "bond3y_change_20d",
    "base_rate_change",
    "rate_spread_3y"
]


MODEL_FEATURES = (
    ASSET_FEATURES
    + MARKET_FEATURES
    + MACRO_FEATURES
)

TARGET = "target_return"


print(
    "feature count:",
    len(MODEL_FEATURES)
)

feature count: 28


In [4]:
# 06a-4. model dataset 품질 확인

print(
    "feature nan:",
    supervised_dataset[
        MODEL_FEATURES
    ]
    .isna()
    .sum()
    .sum()
)

print(
    "feature inf:",
    np.isinf(
        supervised_dataset[
            MODEL_FEATURES
        ]
    )
    .sum()
    .sum()
)

print(
    "target nan:",
    supervised_dataset[
        TARGET
    ]
    .isna()
    .sum()
)

print(
    "target inf:",
    np.isinf(
        supervised_dataset[
            TARGET
        ]
    )
    .sum()
)

feature nan: 0
feature inf: 0
target nan: 0
target inf: 0


In [5]:
# 06a-5. cross-sectional ic 함수

def calculate_ic(
    data,
    prediction_column,
    target_column
):

    ic_values = []

    for _, group in data.groupby(
        "signal_date"
    ):

        if len(group) < 2:
            continue

        if (
            group[prediction_column].nunique() < 2
            or
            group[target_column].nunique() < 2
        ):
            continue

        ic = (
            group[
                prediction_column
            ]
            .rank()
            .corr(
                group[
                    target_column
                ]
                .rank()
            )
        )

        if pd.notna(ic):
            ic_values.append(ic)

    return np.array(
        ic_values
    )

In [6]:
# 06a-6. walk-forward 설정

TRAIN_YEARS = 3
VALIDATION_MONTHS = 6
TEST_MONTHS = 6
STEP_MONTHS = 6

In [7]:
# 06a-7. walk-forward fold 생성

signal_start = (
    supervised_dataset[
        "signal_date"
    ].min()
)

signal_end = (
    supervised_dataset[
        "signal_date"
    ].max()
)


folds = []

validation_start = (
    signal_start
    + pd.DateOffset(
        years=TRAIN_YEARS
    )
)

fold_id = 1


while True:

    test_start = (
        validation_start
        + pd.DateOffset(
            months=VALIDATION_MONTHS
        )
    )

    test_end = (
        test_start
        + pd.DateOffset(
            months=TEST_MONTHS
        )
    )

    if test_end > signal_end:
        break

    folds.append(
        {
            "fold": fold_id,
            "train_start": signal_start,
            "train_end": validation_start,
            "validation_start": validation_start,
            "validation_end": test_start,
            "test_start": test_start,
            "test_end": test_end
        }
    )

    validation_start = (
        validation_start
        + pd.DateOffset(
            months=STEP_MONTHS
        )
    )

    fold_id += 1


fold_table = pd.DataFrame(
    folds
)

print(
    fold_table.to_string(
        index=False
    )
)

 fold train_start  train_end validation_start validation_end test_start   test_end
    1  2018-05-04 2021-05-04       2021-05-04     2021-11-04 2021-11-04 2022-05-04
    2  2018-05-04 2021-11-04       2021-11-04     2022-05-04 2022-05-04 2022-11-04
    3  2018-05-04 2022-05-04       2022-05-04     2022-11-04 2022-11-04 2023-05-04
    4  2018-05-04 2022-11-04       2022-11-04     2023-05-04 2023-05-04 2023-11-04
    5  2018-05-04 2023-05-04       2023-05-04     2023-11-04 2023-11-04 2024-05-04
    6  2018-05-04 2023-11-04       2023-11-04     2024-05-04 2024-05-04 2024-11-04
    7  2018-05-04 2024-05-04       2024-05-04     2024-11-04 2024-11-04 2025-05-04
    8  2018-05-04 2024-11-04       2024-11-04     2025-05-04 2025-05-04 2025-11-04
    9  2018-05-04 2025-05-04       2025-05-04     2025-11-04 2025-11-04 2026-05-04


In [8]:
# 06a-8. ridge alpha 후보 고정

RIDGE_ALPHAS = np.logspace(
    -4,
    8,
    13
)

print(
    RIDGE_ALPHAS
)

[1.e-04 1.e-03 1.e-02 1.e-01 1.e+00 1.e+01 1.e+02 1.e+03 1.e+04 1.e+05
 1.e+06 1.e+07 1.e+08]


Ridge의 alpha는 validation RMSE가 가장 낮은 값으로 선택. 이유는 이 프로젝트의 supervised prediction이 이후 MVO의 expected return(기대수익률) 자체로 들어가기 때문에 단순 순위뿐 아니라 예측 수익률의 크기 오차도 중요하기 때문. Ridge 자체도 squared-error에 L2 penalty를 더한 목적함수를 최소화하는 모델.
IC는 버리지 않고 보조 진단지표로 계속 기록.
또 StandardScaler는 반드시 Pipeline 안에 둠. 그래야 각 fold에서 train 데이터로만 scaler를 학습하고 validation/test에 적용해서 scaling 단계의 정보누출을 막을 수 있음. Ridge는 feature scale에 따라 L2 penalty의 효과가 달라지기 때문에 scaling을 같이 쓰는 게 필요.

In [9]:
# 06a-9. ridge alpha validation 함수

def evaluate_ridge_alphas(
    train_df,
    validation_df
):

    X_train = train_df[
        MODEL_FEATURES
    ]

    y_train = train_df[
        TARGET
    ]

    X_validation = validation_df[
        MODEL_FEATURES
    ]

    y_validation = validation_df[
        TARGET
    ]

    results = []


    for alpha in RIDGE_ALPHAS:

        model = Pipeline(
            [
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "ridge",
                    Ridge(
                        alpha=alpha
                    )
                )
            ]
        )

        model.fit(
            X_train,
            y_train
        )

        prediction = model.predict(
            X_validation
        )


        result_df = (
            validation_df[
                [
                    "signal_date",
                    "ticker",
                    TARGET
                ]
            ]
            .copy()
        )

        result_df[
            "prediction"
        ] = prediction


        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                prediction
            )
        )

        mae = mean_absolute_error(
            y_validation,
            prediction
        )

        r2 = r2_score(
            y_validation,
            prediction
        )


        ic_values = calculate_ic(
            result_df,
            "prediction",
            TARGET
        )


        results.append(
            {
                "alpha": alpha,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "mean_ic": (
                    ic_values.mean()
                    if len(ic_values) > 0
                    else np.nan
                ),
                "median_ic": (
                    np.median(
                        ic_values
                    )
                    if len(ic_values) > 0
                    else np.nan
                ),
                "ic_signals": len(
                    ic_values
                )
            }
        )


    return pd.DataFrame(
        results
    )

train
->
13개 alpha 각각 학습
->
validation에서 RMSE / MAE / R² / IC 확인
->
validation RMSE 최소 alpha 선택

In [10]:
# 06a-10. fold 1 validation

fold = folds[0]


train_mask = (
    (
        supervised_dataset["signal_date"]
        < fold["train_end"]
    )
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["train_end"]
    )
)


validation_mask = (
    (
        supervised_dataset["signal_date"]
        >= fold["validation_start"]
    )
    &
    (
        supervised_dataset["signal_date"]
        < fold["validation_end"]
    )
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["validation_end"]
    )
)


train_df = (
    supervised_dataset.loc[
        train_mask
    ]
    .copy()
)


validation_df = (
    supervised_dataset.loc[
        validation_mask
    ]
    .copy()
)


print(
    "train:",
    train_df.shape
)

print(
    "validation:",
    validation_df.shape
)

train: (7791, 39)
validation: (1248, 39)


next_execution_date <= boundary 조건을 쓰는 이유는 명확함. 예를 들어 train 마지막 signal의 정답이 validation 기간의 미래 가격을 사용한다면 그 정답은 train 종료 시점에는 아직 알 수 없는 정보니까 제외. 시간순 train이 점점 누적되는 expanding-window 구조 자체도 TimeSeriesSplit의 기본 원리와 같음.

In [11]:
# 06a-11. fold 1 alpha 비교

ridge_validation_results = (
    evaluate_ridge_alphas(
        train_df,
        validation_df
    )
)


print(
    ridge_validation_results
    .sort_values(
        [
            "rmse",
            "alpha"
        ]
    )
    .to_string(
        index=False
    )
)

       alpha     rmse      mae        r2   mean_ic  median_ic  ic_signals
1.000000e+05 0.065537 0.044394 -0.000412 -0.047646  -0.020312          25
1.000000e+06 0.065558 0.044376 -0.001069 -0.051911  -0.025402          25
1.000000e+07 0.065563 0.044376 -0.001211 -0.052383  -0.023193          25
1.000000e+08 0.065564 0.044376 -0.001226 -0.052506  -0.023193          25
1.000000e+04 0.065758 0.044925 -0.007182 -0.010391   0.012629          25
1.000000e+03 0.066652 0.046436 -0.034732  0.003301  -0.034526          25
1.000000e+02 0.067432 0.047626 -0.059092 -0.006048  -0.052485          25
1.000000e+01 0.067626 0.047911 -0.065207 -0.005830  -0.051525          25
1.000000e+00 0.067649 0.047943 -0.065927 -0.004424  -0.058343          25
1.000000e-01 0.067651 0.047947 -0.066000 -0.004534  -0.059304          25
1.000000e-02 0.067651 0.047947 -0.066007 -0.004534  -0.059304          25
1.000000e-03 0.067651 0.047947 -0.066008 -0.004534  -0.059304          25
1.000000e-04 0.067651 0.047947 -0.0660

In [12]:
# 06a-12. fold 1 best alpha 선택

best_row = (
    ridge_validation_results
    .sort_values(
        [
            "rmse",
            "alpha"
        ]
    )
    .iloc[0]
)


best_alpha = float(
    best_row[
        "alpha"
    ]
)


print(
    "best alpha:",
    best_alpha
)

print(
    best_row
)

best alpha: 100000.0
alpha         100000.000000
rmse               0.065537
mae                0.044394
r2                -0.000412
mean_ic           -0.047646
median_ic         -0.020312
ic_signals        25.000000
Name: 9, dtype: float64


In [13]:
# 06a-13. ridge walk-forward 실행

ridge_fold_rows = []
ridge_oos_frames = []


for fold in folds:

    train_mask = (
        (supervised_dataset["signal_date"] < fold["train_end"])
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["train_end"]
        )
    )

    validation_mask = (
        (
            supervised_dataset["signal_date"]
            >= fold["validation_start"]
        )
        &
        (
            supervised_dataset["signal_date"]
            < fold["validation_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["validation_end"]
        )
    )

    test_mask = (
        (
            supervised_dataset["signal_date"]
            >= fold["test_start"]
        )
        &
        (
            supervised_dataset["signal_date"]
            < fold["test_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["test_end"]
        )
    )

    train_df = (
        supervised_dataset.loc[
            train_mask
        ]
        .copy()
    )

    validation_df = (
        supervised_dataset.loc[
            validation_mask
        ]
        .copy()
    )

    test_df = (
        supervised_dataset.loc[
            test_mask
        ]
        .copy()
    )


    validation_results = (
        evaluate_ridge_alphas(
            train_df,
            validation_df
        )
    )

    best_row = (
        validation_results
        .sort_values(
            [
                "rmse",
                "alpha"
            ]
        )
        .iloc[0]
    )

    best_alpha = float(
        best_row["alpha"]
    )


    train_validation_mask = (
        (
            supervised_dataset["signal_date"]
            < fold["test_start"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["test_start"]
        )
    )

    train_validation_df = (
        supervised_dataset.loc[
            train_validation_mask
        ]
        .copy()
    )


    X_train_validation = (
        train_validation_df[
            MODEL_FEATURES
        ]
    )

    y_train_validation = (
        train_validation_df[
            TARGET
        ]
    )

    X_test = (
        test_df[
            MODEL_FEATURES
        ]
    )

    y_test = (
        test_df[
            TARGET
        ]
    )


    ridge_model = Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),
            (
                "ridge",
                Ridge(
                    alpha=best_alpha
                )
            )
        ]
    )

    ridge_model.fit(
        X_train_validation,
        y_train_validation
    )

    ridge_pred = ridge_model.predict(
        X_test
    )


    dummy_model = DummyRegressor(
        strategy="mean"
    )

    dummy_model.fit(
        X_train_validation,
        y_train_validation
    )

    dummy_pred = dummy_model.predict(
        X_test
    )


    test_result = (
        test_df[
            [
                "signal_date",
                "execution_date",
                "ticker",
                "name",
                TARGET
            ]
        ]
        .copy()
    )

    test_result["prediction"] = (
        ridge_pred
    )

    test_result["dummy_prediction"] = (
        dummy_pred
    )

    test_result["fold"] = (
        fold["fold"]
    )


    ic_values = calculate_ic(
        test_result,
        "prediction",
        TARGET
    )


    ridge_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            ridge_pred
        )
    )

    ridge_mae = mean_absolute_error(
        y_test,
        ridge_pred
    )

    ridge_r2 = r2_score(
        y_test,
        ridge_pred
    )


    dummy_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            dummy_pred
        )
    )

    dummy_mae = mean_absolute_error(
        y_test,
        dummy_pred
    )

    dummy_r2 = r2_score(
        y_test,
        dummy_pred
    )


    ridge_fold_rows.append(
        {
            "fold": fold["fold"],
            "best_alpha": best_alpha,
            "test_start": fold["test_start"],
            "test_end": fold["test_end"],
            "test_rows": len(test_df),
            "test_signals": test_df[
                "signal_date"
            ].nunique(),
            "ridge_rmse": ridge_rmse,
            "ridge_mae": ridge_mae,
            "ridge_r2": ridge_r2,
            "mean_ic": (
                ic_values.mean()
                if len(ic_values) > 0
                else np.nan
            ),
            "median_ic": (
                np.median(
                    ic_values
                )
                if len(ic_values) > 0
                else np.nan
            ),
            "dummy_rmse": dummy_rmse,
            "dummy_mae": dummy_mae,
            "dummy_r2": dummy_r2
        }
    )

    ridge_oos_frames.append(
        test_result
    )

여기서 각 fold마다:
train
→ validation에서 alpha 선택
→ alpha 고정
→ train + validation 재학습
→ test 반복

test 결과는 다음 fold의 alpha 선택에 사용하지 x

In [14]:
# 06a-14. ridge fold 결과 생성

ridge_fold_results = pd.DataFrame(
    ridge_fold_rows
)

ridge_fold_results[
    "rmse_improvement"
] = (
    ridge_fold_results[
        "dummy_rmse"
    ]
    - ridge_fold_results[
        "ridge_rmse"
    ]
)

ridge_fold_results[
    "ridge_better"
] = (
    ridge_fold_results[
        "ridge_rmse"
    ]
    <
    ridge_fold_results[
        "dummy_rmse"
    ]
)


print(
    ridge_fold_results.to_string(
        index=False
    )
)

 fold  best_alpha test_start   test_end  test_rows  test_signals  ridge_rmse  ridge_mae  ridge_r2   mean_ic  median_ic  dummy_rmse  dummy_mae  dummy_r2  rmse_improvement  ridge_better
    1    100000.0 2021-11-04 2022-05-04       1250            25    0.064162   0.047195 -0.013011 -0.057940  -0.078127    0.064548   0.047621 -0.025234      3.859379e-04          True
    2     10000.0 2022-05-04 2022-11-04       1250            25    0.065222   0.047433 -0.018258 -0.072346  -0.071789    0.065276   0.047685 -0.019936      5.371359e-05          True
    3    100000.0 2022-11-04 2023-05-04       1248            25    0.074754   0.049259 -0.016305 -0.045032  -0.091285    0.074470   0.049080 -0.008575     -2.848387e-04         False
    4 100000000.0 2023-05-04 2023-11-04       1250            25    0.076923   0.050796 -0.008782 -0.092479  -0.059784    0.076923   0.050796 -0.008781     -5.526336e-08         False
    5 100000000.0 2023-11-04 2024-05-04       1197            24    0.076069   0

In [15]:
# 06a-15. ridge oos prediction 결합

ridge_oos_predictions = (
    pd.concat(
        ridge_oos_frames,
        ignore_index=True
    )
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "shape:",
    ridge_oos_predictions.shape
)

print(
    "signals:",
    ridge_oos_predictions[
        "signal_date"
    ].nunique()
)

print(
    "start:",
    ridge_oos_predictions[
        "signal_date"
    ].min()
)

print(
    "end:",
    ridge_oos_predictions[
        "signal_date"
    ].max()
)

print(
    "duplicates:",
    ridge_oos_predictions[
        [
            "signal_date",
            "ticker"
        ]
    ]
    .duplicated()
    .sum()
)

shape: (11140, 8)
signals: 223
start: 2021-11-05 00:00:00
end: 2026-04-24 00:00:00
duplicates: 0


In [16]:
# 06a-16. ridge 전체 oos 성능

pooled_rmse = np.sqrt(
    mean_squared_error(
        ridge_oos_predictions[
            TARGET
        ],
        ridge_oos_predictions[
            "prediction"
        ]
    )
)

pooled_mae = mean_absolute_error(
    ridge_oos_predictions[
        TARGET
    ],
    ridge_oos_predictions[
        "prediction"
    ]
)

pooled_r2 = r2_score(
    ridge_oos_predictions[
        TARGET
    ],
    ridge_oos_predictions[
        "prediction"
    ]
)

pooled_ic = calculate_ic(
    ridge_oos_predictions,
    "prediction",
    TARGET
)


print(
    "pooled rmse:",
    pooled_rmse
)

print(
    "pooled mae:",
    pooled_mae
)

print(
    "pooled r2:",
    pooled_r2
)

print(
    "mean ic:",
    pooled_ic.mean()
)

print(
    "median ic:",
    np.median(
        pooled_ic
    )
)

print(
    "ridge better folds:",
    ridge_fold_results[
        "ridge_better"
    ].sum(),
    "/",
    len(
        ridge_fold_results
    )
)

pooled rmse: 0.07860736765071763
pooled mae: 0.05445240182660251
pooled r2: -0.0005980733309269848
mean ic: -0.06879394807250255
median ic: -0.07649459783913565
ridge better folds: 4 / 9


fold mean
= 각 6개월 구간을 동일한 비중으로 평가

pooled
= 모든 OOS sample을 한꺼번에 평가

In [17]:
# 06a-17. ridge prediction 분산 확인

prediction_stats = (
    ridge_oos_predictions
    .groupby(
        "signal_date"
    )
    .agg(
        prediction_std=(
            "prediction",
            "std"
        ),
        target_std=(
            TARGET,
            "std"
        )
    )
)


print(
    prediction_stats.describe()
)

       prediction_std  target_std
count    2.230000e+02  223.000000
mean     2.416017e-04    0.068816
std      3.875072e-04    0.020857
min      3.993135e-07    0.034853
25%      8.160068e-07    0.051816
50%      1.234962e-06    0.064346
75%      4.318000e-04    0.082868
max      1.905782e-03    0.137248


전체 OOS에서 R²=-0.0006, Dummy보다 나은 fold가 4/9, 평균 IC가 -0.0688\
특히 실제 종목별 주간 수익률의 평균 표준편차가 약 6.88%인데 Ridge 예측의 종목별 표준편차는 약 0.024%밖에 안 됌. 실제 분산보다 약 285배 작게 예측. 즉 Ridge는 많은 기간에 사실상 종목별 차이를 거의 만들지 못하고 평균예측 쪽으로 수축하고 있다는 해석이 데이터와 일치함.

또 4·5·6·8·9번 fold에서 alpha=1e8이라는 상한값이 선택됐음. 정확한 최적 alpha가 1e8보다 더 큰지는 알 수 없지만, Ridge는 alpha가 커질수록 계수가 0에 가까워져 평균예측에 접근하므로, 지금 OOS 결과를 본 뒤 다시 search range를 확장하지 않고 “강한 regularization 경계 선택”이라는 결과를 저장함.

In [18]:
# 06a-18. ridge 결과 저장 경로

RIDGE_RESULT_DIR = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "06a_ridge"
)

RIDGE_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RIDGE_FOLD_PATH = (
    RIDGE_RESULT_DIR
    / "ridge_fold_results.csv"
)

RIDGE_OOS_PATH = (
    RIDGE_RESULT_DIR
    / "ridge_oos_predictions.parquet"
)

In [19]:
# 06a-19. alpha boundary 기록

ridge_fold_results["alpha_upper_bound"] = (
    ridge_fold_results["best_alpha"]
    == RIDGE_ALPHAS.max()
)

print(
    ridge_fold_results[
        [
            "fold",
            "best_alpha",
            "alpha_upper_bound"
        ]
    ]
)

   fold   best_alpha  alpha_upper_bound
0     1     100000.0              False
1     2      10000.0              False
2     3     100000.0              False
3     4  100000000.0               True
4     5  100000000.0               True
5     6  100000000.0               True
6     7    1000000.0              False
7     8  100000000.0               True
8     9  100000000.0               True


In [20]:
# 06a-20. ridge 결과 저장

ridge_fold_results.to_csv(
    RIDGE_FOLD_PATH,
    index=False,
    encoding="utf-8-sig"
)

ridge_oos_table = pa.Table.from_pandas(
    ridge_oos_predictions,
    preserve_index=False
)

pq.write_table(
    ridge_oos_table,
    RIDGE_OOS_PATH,
    compression="snappy"
)

print(
    "saved:",
    RIDGE_FOLD_PATH
)

print(
    "saved:",
    RIDGE_OOS_PATH
)

saved: C:\code\portfolio_optimization\data\predictions\06a_ridge\ridge_fold_results.csv
saved: C:\code\portfolio_optimization\data\predictions\06a_ridge\ridge_oos_predictions.parquet
